# Multiclass Dataset Preprocessing

This notebook preprocesses the multiclass dataset to create files compatible with the benchmark framework.
The multiclass dataset contains images with multiple object classes, where each image can have objects from different categories.

## Data Structure
- **annotations.json**: Contains image annotations with points and corresponding class IDs
- **class_mapping.json**: Maps class IDs to class names

## Output Files
- **multiclass_image_classes.txt**: Image-to-primary-class mapping (for compatibility)
- **multiclass_gt_counts.json**: Ground truth counts per image and class
- **multiclass_split_classes.json**: All available classes
- **multiclass_split_images.json**: All images for testing
- **multiclass_class_counts_per_image.json**: Detailed class counts per image

In [13]:
import json
import os
from collections import defaultdict, Counter
import pandas as pd
import numpy as np

In [14]:
# Load the annotations and class mapping
with open('annotations.json', 'r') as f:
    annotations = json.load(f)

with open('class_mapping.json', 'r') as f:
    class_mapping = json.load(f)

print(f"Total images: {len(annotations)}")
print(f"Total classes: {len(class_mapping)}")
print("\nFirst 5 class mappings:")
for i, (k, v) in enumerate(class_mapping.items()):
    if i < 5:
        print(f"  {k}: {v}")

Total images: 200
Total classes: 45

First 5 class mappings:
  0: plates
  1: bowls
  2: cups
  3: forks
  4: glasses


In [24]:
# Analyze the dataset structure
total_objects = 0
classes_per_image = []
class_distribution = defaultdict(int)
images_per_class = defaultdict(list)

for img_name, data in annotations.items():
    points = data['points']
    classes = data['classes']
    
    total_objects += len(points)
    unique_classes = set(classes)
    classes_per_image.append(len(unique_classes))
    
    # Count class occurrences
    class_counts = Counter(classes)
    for class_id, count in class_counts.items():
        class_distribution[class_id] += count
        images_per_class[class_id].append(img_name)

    if len(unique_classes) == 1:
        print(f"Image {img_name} has only one class!")

print(f"Total objects across all images: {total_objects}")
print(f"Average classes per image: {np.mean(classes_per_image):.2f}")
print(f"Max classes per image: {max(classes_per_image)}")
print(f"Min classes per image: {min(classes_per_image)}")

Total objects across all images: 11576
Average classes per image: 2.55
Max classes per image: 7
Min classes per image: 2


In [16]:
# Create class distribution summary
print("Class distribution (class_name: total_objects, images_with_class):")
for class_id in sorted(class_distribution.keys()):
    class_name = class_mapping[str(class_id)]
    total_objects_of_class = class_distribution[class_id]
    images_with_class = len(images_per_class[class_id])
    print(f"  {class_name}: {total_objects_of_class} objects, {images_with_class} images")

Class distribution (class_name: total_objects, images_with_class):
  plates: 168 objects, 13 images
  bowls: 36 objects, 9 images
  cups: 26 objects, 9 images
  forks: 90 objects, 11 images
  glasses: 17 objects, 7 images
  knives: 68 objects, 11 images
  chairs: 314 objects, 15 images
  spoons: 62 objects, 8 images
  cherries: 573 objects, 6 images
  raspberries: 291 objects, 14 images
  strawberries: 61 objects, 8 images
  blueberries: 627 objects, 14 images
  blackberries: 223 objects, 14 images
  red currants: 301 objects, 6 images
  grapes: 208 objects, 3 images
  lemons: 219 objects, 11 images
  apples: 242 objects, 18 images
  pears: 37 objects, 10 images
  bananas: 22 objects, 6 images
  oranges: 94 objects, 6 images
  people: 3756 objects, 60 images
  beach umbrellas: 675 objects, 10 images
  cars: 959 objects, 40 images
  motorbikes: 472 objects, 25 images
  bicycles: 17 objects, 8 images
  trucks: 66 objects, 16 images
  amplifiers: 127 objects, 13 images
  guitars: 323 obje

In [17]:
# Create multiclass_class_counts_per_image.json - detailed counts for each class in each image
multiclass_class_counts = {}

for img_name, data in annotations.items():
    classes = data['classes']
    class_counts = Counter(classes)
    
    # Convert class IDs to class names and ensure all classes are represented
    img_class_counts = {}
    for class_id_str, class_name in class_mapping.items():
        class_id = int(class_id_str)
        img_class_counts[class_name] = class_counts.get(class_id, 0)
    
    multiclass_class_counts[img_name] = img_class_counts

# Save the detailed class counts
with open('multiclass_class_counts_per_image.json', 'w') as f:
    json.dump(multiclass_class_counts, f, indent=2)

print("Created multiclass_class_counts_per_image.json")
print("Sample entry:")
sample_img = list(multiclass_class_counts.keys())[0]
print(f"  {sample_img}: {multiclass_class_counts[sample_img]}")

Created multiclass_class_counts_per_image.json
Sample entry:
  mcac_1.jpg: {'plates': 7, 'bowls': 13, 'cups': 0, 'forks': 0, 'glasses': 0, 'knives': 0, 'chairs': 0, 'spoons': 0, 'cherries': 0, 'raspberries': 0, 'strawberries': 0, 'blueberries': 0, 'blackberries': 0, 'red currants': 0, 'grapes': 0, 'lemons': 0, 'apples': 0, 'pears': 0, 'bananas': 0, 'oranges': 0, 'people': 0, 'beach umbrellas': 0, 'cars': 0, 'motorbikes': 0, 'bicycles': 0, 'trucks': 0, 'amplifiers': 0, 'guitars': 0, 'cats': 0, 'dogs': 0, 'balconies': 0, 'windows': 0, 'desks': 0, 'cigaretts': 0, 'lighters': 0, 'keys': 0, 'padlocks': 0, 'key holders': 0, 'cows': 0, 'sheeps': 0, 'goats': 0, 'boats': 0, 'pens': 0, 'rulers': 0, 'pencils': 0}


In [18]:
# Create multiclass_gt_counts.json - total object count per image (like FSC147 format)
multiclass_gt_counts = {}

for img_name, data in annotations.items():
    total_objects = len(data['points'])
    multiclass_gt_counts[img_name] = total_objects

# Save ground truth counts
with open('multiclass_gt_counts.json', 'w') as f:
    json.dump(multiclass_gt_counts, f, indent=2)

print("Created multiclass_gt_counts.json")
print(f"Sample entries: {dict(list(multiclass_gt_counts.items())[:3])}")

Created multiclass_gt_counts.json
Sample entries: {'mcac_1.jpg': 20, 'mcac_2.jpg': 18, 'mcac_3.jpg': 14}


In [19]:
# Create multiclass_image_classes.txt - include ALL classes present in each image
# Classes will be sorted by frequency (most common first) and separated by commas
multiclass_image_classes = {}

with open('multiclass_image_classes.txt', 'w') as f:
    for img_name, data in annotations.items():
        classes = data['classes']
        class_counts = Counter(classes)
        
        # Get all classes present in the image, sorted by frequency (most common first)
        sorted_classes = [class_mapping[str(class_id)] for class_id, count in class_counts.most_common()]
        classes_str = ','.join(sorted_classes)
        
        multiclass_image_classes[img_name] = sorted_classes
        f.write(f"{img_name}\t{classes_str}\n")

print("Created multiclass_image_classes.txt with ALL classes per image")
print(f"Sample entries:")
for i, (img_name, classes) in enumerate(list(multiclass_image_classes.items())[:3]):
    print(f"  {img_name}: {classes}")
    if i >= 2:
        break

Created multiclass_image_classes.txt with ALL classes per image
Sample entries:
  mcac_1.jpg: ['bowls', 'plates']
  mcac_2.jpg: ['plates', 'cups']
  mcac_3.jpg: ['plates', 'cups']


In [20]:
# Create multiclass_split_classes.json - all available classes
all_class_names = list(class_mapping.values())
multiclass_split_classes = {
    "test": all_class_names  # Using all data for testing as requested
}

with open('multiclass_split_classes.json', 'w') as f:
    json.dump(multiclass_split_classes, f, indent=2)

print("Created multiclass_split_classes.json")
print(f"Total classes: {len(all_class_names)}")
print(f"Classes: {all_class_names}")

Created multiclass_split_classes.json
Total classes: 45
Classes: ['plates', 'bowls', 'cups', 'forks', 'glasses', 'knives', 'chairs', 'spoons', 'cherries', 'raspberries', 'strawberries', 'blueberries', 'blackberries', 'red currants', 'grapes', 'lemons', 'apples', 'pears', 'bananas', 'oranges', 'people', 'beach umbrellas', 'cars', 'motorbikes', 'bicycles', 'trucks', 'amplifiers', 'guitars', 'cats', 'dogs', 'balconies', 'windows', 'desks', 'cigaretts', 'lighters', 'keys', 'padlocks', 'key holders', 'cows', 'sheeps', 'goats', 'boats', 'pens', 'rulers', 'pencils']


In [21]:
# Create multiclass_split_images.json - all images for testing
all_image_names = list(annotations.keys())
multiclass_split_images = {
    "test": all_image_names  # Using all data for testing as requested
}

with open('multiclass_split_images.json', 'w') as f:
    json.dump(multiclass_split_images, f, indent=2)

print("Created multiclass_split_images.json")
print(f"Total images for testing: {len(all_image_names)}")
print(f"Sample images: {all_image_names[:5]}")

Created multiclass_split_images.json
Total images for testing: 200
Sample images: ['mcac_1.jpg', 'mcac_2.jpg', 'mcac_3.jpg', 'mcac_4.jpg', 'mcac_6.jpg']


In [25]:
# Create summary statistics
print("\n" + "="*50)
print("MULTICLASS DATASET PREPROCESSING SUMMARY")
print("="*50)
print(f"Total images: {len(annotations)}")
print(f"Total classes: {len(class_mapping)}")
print(f"Total objects: {total_objects}")
print(f"Average objects per image: {total_objects / len(annotations):.2f}")
print(f"Average classes per image: {np.mean(classes_per_image):.2f}")

print("\nFiles created:")
print("  - multiclass_image_classes.txt")
print("  - multiclass_gt_counts.json")
print("  - multiclass_split_classes.json")
print("  - multiclass_split_images.json")
print("  - multiclass_class_counts_per_image.json")

print("\nDataset is ready for benchmark testing!")


MULTICLASS DATASET PREPROCESSING SUMMARY
Total images: 200
Total classes: 45
Total objects: 11576
Average objects per image: 57.88
Average classes per image: 2.55

Files created:
  - multiclass_image_classes.txt
  - multiclass_gt_counts.json
  - multiclass_split_classes.json
  - multiclass_split_images.json
  - multiclass_class_counts_per_image.json

Dataset is ready for benchmark testing!
